In [10]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, SVD, accuracy

data_path = "../../data/processed/ratings_cleaned.csv"
print(f"[INFO] Đang nạp dữ liệu từ: {data_path}")
ratings_df = pd.read_csv(data_path)

ratings_df["User-ID"] = ratings_df["User-ID"].astype(int)
ratings_df["ISBN"] = ratings_df["ISBN"].astype(str)
ratings_df["Book-Rating"] = ratings_df["Book-Rating"].astype(float)

total_records = len(ratings_df)
print(f"[INFO] Tổng số bản ghi đánh giá: {total_records}")
ratings_df.head()

[INFO] Đang nạp dữ liệu từ: ../../data/processed/ratings_cleaned.csv
[INFO] Tổng số bản ghi đánh giá: 383842


,User-ID,ISBN,Book-Rating
0,276726,0155061224,5.0
1,276729,052165615X,3.0
2,276729,0521795028,6.0
3,276744,038550120X,7.0
4,276747,0060517794,9.0


In [11]:
train_val_df, test_df = train_test_split(
    ratings_df, 
    test_size=0.20, 
    random_state=42, 
    shuffle=True
)

train_df, val_df = train_test_split(
    train_val_df, 
    test_size=0.25, 
    random_state=42, 
    shuffle=True
)

print(f"Train: {len(train_df)} dòng ({len(train_df)/total_records*100:.1f}%)")
print(f"Val:   {len(val_df)} dòng ({len(val_df)/total_records*100:.1f}%)")
print(f"Test:  {len(test_df)} dòng ({len(test_df)/total_records*100:.1f}%)")

Train: 230304 dòng (60.0%)
Val:   76769 dòng (20.0%)
Test:  76769 dòng (20.0%)


In [12]:
min_rating = ratings_df["Book-Rating"].min()
max_rating = ratings_df["Book-Rating"].max()
reader = Reader(rating_scale=(min_rating, max_rating))

train_data = Dataset.load_from_df(train_df[["User-ID", "ISBN", "Book-Rating"]], reader)
trainset = train_data.build_full_trainset()

valset = list(val_df[["User-ID", "ISBN", "Book-Rating"]].itertuples(index=False, name=None))
testset = list(test_df[["User-ID", "ISBN", "Book-Rating"]].itertuples(index=False, name=None))

print("Đã chuẩn bị xong dữ liệu dạng Surprise trainset, valset và testset.")

Đã chuẩn bị xong dữ liệu dạng Surprise trainset, valset và testset.


In [13]:
print("Đang huấn luyện mô hình SVD...")
model = SVD(
    n_factors=50,       # Số chiều tiềm ẩn (latent factors)
    n_epochs=20,        # Số vòng lặp tối ưu
    lr_all=0.005,       # Learning rate
    reg_all=0.02,       # Regularization chống overfitting
    random_state=42
)

model.fit(trainset)
print("Huấn luyện hoàn tất!")

Đang huấn luyện mô hình SVD...
Huấn luyện hoàn tất!


In [14]:
print("--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (20%) ---")
val_predictions = model.test(valset)
val_rmse = accuracy.rmse(val_predictions)
val_mae = accuracy.mae(val_predictions)

print("\n--- ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST (20%) ---")
test_predictions = model.test(testset)
test_rmse = accuracy.rmse(test_predictions)
test_mae = accuracy.mae(test_predictions)

--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (20%) ---
RMSE: 1.6391
MAE:  1.2658

--- ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST (20%) ---
RMSE: 1.6446
MAE:  1.2696


In [15]:
save_dir = "../saved_models"
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "collaborative_model.pkl")

joblib.dump(model, model_path)
print(f"Đã lưu model Collaborative Filtering thành công tại: {model_path}")

loaded_model = joblib.load(model_path)
sample_user = int(train_df["User-ID"].iloc[0])
sample_isbn = str(train_df["ISBN"].iloc[0])
sample_pred = loaded_model.predict(sample_user, sample_isbn)

print(f"\n[KIỂM THỬ LOAD LẠI]")
print(f"User-ID: {sample_user} | ISBN: {sample_isbn}")
print(f"Điểm dự đoán (Predicted Rating): {sample_pred.est:.2f}")

Đã lưu model Collaborative Filtering thành công tại: ../saved_models\collaborative_model.pkl

[KIỂM THỬ LOAD LẠI]
User-ID: 166596 | ISBN: 0821409794
Điểm dự đoán (Predicted Rating): 7.83


In [16]:
# Test: Hàm hiển thị gợi ý trực tiếp trong Notebook
books_df = pd.read_csv("../../data/processed/books_cleaned.csv")

def recommend_books_for_user(user_id, n_recommendations=5):
    # Lấy sách user đã đọc để bỏ qua
    read_books = ratings_df[ratings_df["User-ID"] == user_id]["ISBN"].unique()
    all_books = books_df["ISBN"].unique()
    unread_books = [book for book in all_books if book not in read_books]
    
    # Dự đoán điểm cho các cuốn chưa đọc
    predictions = [model.predict(user_id, isbn) for isbn in unread_books]
    
    # Lấy top N cuốn có điểm dự đoán cao nhất
    top_preds = sorted(predictions, key=lambda x: x.est, reverse=True)[:n_recommendations]
    
    # Trích xuất thông tin hiển thị bảng
    recommended_isbns = [pred.iid for pred in top_preds]
    result_df = books_df[books_df["ISBN"].isin(recommended_isbns)].copy()
    
    # Gán thêm cột điểm dự đoán
    score_dict = {pred.iid: round(pred.est, 2) for pred in top_preds}
    result_df["Predicted_Rating"] = result_df["ISBN"].map(score_dict)
    
    # Sắp xếp hiển thị theo điểm từ cao xuống thấp
    cols = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Predicted_Rating"]
    available_cols = [c for c in cols if c in result_df.columns]
    return result_df[available_cols].sort_values(by="Predicted_Rating", ascending=False)

# Chạy thử kiểm tra với chính User vừa test
recommend_books_for_user(user_id=166596, n_recommendations=5)

C:\Users\Huy\AppData\Local\Temp\ipykernel_10080\2272454786.py:2: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv("../../data/processed/books_cleaned.csv")


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Predicted_Rating
1399,006092988X,A Tree Grows in Brooklyn,Betty Smith,1998,9.36
2143,059035342X,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,1999,9.35
5431,0439139597,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,2000,9.30
9026,043936213X,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,2001,9.26
6330,0439136369,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,2001,9.23
